# HCMUTE Chatbot – 3-Pipeline Evaluation + Ablation

**Main pipelines** (all use **gpt-5-mini**):
1. **LLM Only** – No retrieval, pure generation
2. **Basic RAG** – Dense retrieval → Generate
3. **Our RAG** – Tool routing → Query expansion → RRF → Rerank → Text2SQL → Generate

**Ablation** (remove one component at a time from Our RAG):

4. **− Reranker** – No Jina reranking
5. **− Query Expansion** – Single query, no multi-query + RRF
6. **− Text2SQL** – Tool routing can only pick document_search
7. **− Tool Routing** – Skips LLM tool selection, always document_search

**Controlled experiment**: Same model, same collection, same dataset. Only the pipeline architecture varies.

**Two-round latency**: Round 1 = cold, Round 2 = cached.

Output: `output/evaluation_results.csv` → then run `judge.ipynb` for scoring & statistics.

In [1]:
import pandas as pd
import time
import asyncio
import json
import os
from tqdm.auto import tqdm
from langchain_qdrant import RetrievalMode

from config import (
    settings,
    SEED,
    GENERATION_MODEL,
    LLM_ONLY_MODEL,
    RAG_COLLECTION,
    RETRIEVAL_K,
    RERANKER_TOP_K,
    RERANKER_MODEL,
    FULL_DATASET_PATH_DIEMCHUAN,
    FULL_DATASET_PATH_TUYENSINH,
    CORPUS_PATH,
    OUTPUT_PATH_DIEMCHUAN,
    OUTPUT_PATH_TUYENSINH,
    OUTPUT_PATH_TUYENSINH_ABLATION,
    OUTPUT_PATH_DIEMCHUAN_ABLATION,
    PIPELINE_LABELS,
    CACHE_THRESHOLD,
)
from components.vector_store import get_vector_store
from components.embeddings import get_dense_embedding_model
from components.reranks import JinaReranker
from components.cache import SemanticCache
from components.pipelines.llm_only import LLMOnlyPipeline
from components.pipelines.basic_rag import BasicRAGPipeline
from components.pipelines.our_rag import OurRAGPipeline

LIST_COLUMNS = ["context", "retrieved_doc_contents"]

def parse_list_columns(df: pd.DataFrame) -> pd.DataFrame:
    for col in LIST_COLUMNS:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: json.loads(x) if isinstance(x, str) and x.strip() else []
            )
    return df

def serialize_list_columns(df: pd.DataFrame) -> pd.DataFrame:
    save_df = df.copy()
    for col in LIST_COLUMNS:
        save_df[col] = save_df[col].apply(
            lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else "[]"
        )
    return save_df

print("Imports OK")

dense_embedding: client=<openai.resources.embeddings.Embeddings object at 0x7feac931edd0> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7feac3a50210> model='text-embedding-3-small' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True
Imports OK


In [6]:
shared_vs_dense = get_vector_store(
    mode=RetrievalMode.DENSE,
    collection_name=RAG_COLLECTION,
)
shared_vs_hybrid = get_vector_store(
    mode=RetrievalMode.HYBRID,
    collection_name=RAG_COLLECTION,
)

llm_only = LLMOnlyPipeline(
    model_name=LLM_ONLY_MODEL["model_name"],
    temperature=LLM_ONLY_MODEL["temperature"],
    seed=LLM_ONLY_MODEL.get("seed"),
)

basic_rag = BasicRAGPipeline(
    vector_store=shared_vs_dense,
    k=RETRIEVAL_K,
    model_name=GENERATION_MODEL["model_name"],
    temperature=GENERATION_MODEL["temperature"],
    reranker=None,
    seed=GENERATION_MODEL.get("seed"),
)

reranker = JinaReranker(model_id=RERANKER_MODEL)
_common = dict(
    vector_store=shared_vs_hybrid,
    k=RETRIEVAL_K,
    model_name=GENERATION_MODEL["model_name"],
    temperature=GENERATION_MODEL["temperature"],
    seed=GENERATION_MODEL.get("seed"),
)

our_rag = OurRAGPipeline(**_common, reranker=reranker, rerank_top_k=RERANKER_TOP_K)

pipelines = {
    # "llm_only":             llm_only,
    # "basic_rag":            basic_rag,
    # "our_rag":              our_rag,
    # "ablation_no_rerank":   OurRAGPipeline(**_common, reranker=None),
    "ablation_no_qe":       OurRAGPipeline(**_common, reranker=reranker,
                                           rerank_top_k=RERANKER_TOP_K, use_query_expansion=False),
    # "ablation_no_text2sql": OurRAGPipeline(**_common, reranker=reranker,
    #                                        rerank_top_k=RERANKER_TOP_K, disable_text2sql=True),
    # "ablation_no_routing":  OurRAGPipeline(**_common, reranker=reranker,
    #                                        rerank_top_k=RERANKER_TOP_K, use_tool_routing=False),
}

embedding_model = get_dense_embedding_model()
semantic_cache = SemanticCache(embeddings=embedding_model, threshold=CACHE_THRESHOLD)

print(f"Initialized {len(pipelines)} pipelines: {list(pipelines.keys())}")
print(f"Shared collection: {RAG_COLLECTION}")
print(f"Seed: {SEED}")
print(f"Semantic cache: threshold={CACHE_THRESHOLD}")

Initialized 1 pipelines: ['ablation_no_qe']
Shared collection: method_naive_chunks_chunk_size_1024_chunk_overlap_128_hybrid
Seed: 42
Semantic cache: threshold=0.9


In [7]:
df_tuyensinh = pd.read_csv(FULL_DATASET_PATH_TUYENSINH)
df_diemchuan = pd.read_csv(FULL_DATASET_PATH_DIEMCHUAN)

In [9]:
df_tuyensinh.head()

,question,answer
0,Trường mình tên đầy đủ là gì vậy ạ?,Tên đầy đủ của trường mình là Trường Đại học S...
1,Địa chỉ chính xác của trường ở đâu thế ạ?,"Trường tọa lạc tại số 01, Võ Văn Ngân, Phường ..."
2,Trường được thành lập vào ngày nào vậy ạ?,Trường chúng ta chính thức được thành lập vào ...
3,Tiền thân của trường mình là đơn vị nào?,Tiền thân của trường mình chính là Ban Cao đẳn...
4,Giá trị cốt lõi của trường gồm những gì ạ?,Các giá trị cốt lõi của trường bao gồm Nhân bả...


In [10]:
df_diemchuan.head()

,question,answer
0,Cho tôi hỏi điểm trúng tuyển của ngành Sư phạm...,"Theo dữ liệu tuyển sinh, điểm trúng tuyển ngàn..."
1,Điểm chuẩn ngành Công nghệ Kỹ thuật ô tô (đào ...,"Theo nguồn tin, điểm chuẩn của ngành Công nghệ..."
2,"Năm 2024, ngành Công nghệ thông tin (CTĐT Tiến...","Trong năm 2024, điểm trúng tuyển của ngành Côn..."
3,Xin cho biết điểm chuẩn ngành Robot và trí tuệ...,Điểm trúng tuyển vào ngành Robot và trí tuệ nh...
4,Ngành Thương mại điện tử (đào tạo bằng tiếng V...,"Theo dữ liệu, mức điểm trúng tuyển của ngành T..."


## Tạo evaluation dataset cho điểm chuẩn

In [11]:
print(f"Loaded {len(df_diemchuan)} questions from {FULL_DATASET_PATH_DIEMCHUAN}")

corpus_df = pd.read_csv(CORPUS_PATH)
corpus_lookup = {idx: row["name"] + "\n" + row["full_text"] for idx, row in corpus_df.iterrows()}
print(f"Loaded {len(corpus_lookup)} corpus documents for doc_id → full text lookup")

semantic_cache.clear()

results = []

for pipe_key, pipe in pipelines.items():
    label = PIPELINE_LABELS[pipe_key]
    print(f"\n{'='*60}")
    print(f"[Round 1 – Cold] Running pipeline: {label}")
    print(f"{'='*60}")

    for idx, row in tqdm(df_diemchuan.iterrows(), total=len(df_diemchuan), desc=label):
        question = row["question"]
        ground_truth = row["answer"]

        t0 = time.perf_counter()
        try:
            result = await pipe.run(question)
            latency = time.perf_counter() - t0
            answer = result.answer
            context = result.context
            doc_ids = result.doc_ids
            tool_used = result.tool_used
        except Exception as e:
            latency = time.perf_counter() - t0
            answer = f"ERROR: {e}"
            context = []
            doc_ids = []
            tool_used = "error"

        unique_ids = list(dict.fromkeys(doc_ids))
        retrieved_doc_contents = [
            corpus_lookup[int(did)] for did in unique_ids if int(did) in corpus_lookup
        ]

        await semantic_cache.add(question, answer)

        results.append({
            "qid": idx,
            "pipeline": pipe_key,
            "question": question,
            "ground_truth_answer": ground_truth,
            "generated_answer": answer,
            "context": context,
            "retrieved_doc_contents": retrieved_doc_contents,
            "tool_used": tool_used,
            "latency_s": round(latency, 3),
            "latency_cached_s": None,
        })

results_df_diemchuan = pd.DataFrame(results)
os.makedirs("output", exist_ok=True)

serialize_list_columns(results_df_diemchuan).to_csv(OUTPUT_PATH_DIEMCHUAN_ABLATION, index=False, encoding="utf-8")
print(f"\nRound 1 done – saved {len(results_df_diemchuan)} rows to {OUTPUT_PATH_DIEMCHUAN_ABLATION}")
print(f"Semantic cache populated with {semantic_cache.size} entries")
results_df_diemchuan.head()

Loaded 200 questions from dataset/evaluation_diemchuan.csv
Loaded 498 corpus documents for doc_id → full text lookup

[Round 1 – Cold] Running pipeline: − Query Expansion


− Query Expansion:   0%|          | 0/200 [00:00<?, ?it/s]


Round 1 done – saved 200 rows to output/ablation_query_expansion_diemchuan.csv
Semantic cache populated with 200 entries


,qid,pipeline,question,ground_truth_answer,generated_answer,context,retrieved_doc_contents,tool_used,latency_s,latency_cached_s
0,0,ablation_no_qe,Cho tôi hỏi điểm trúng tuyển của ngành Sư phạm...,"Theo dữ liệu tuyển sinh, điểm trúng tuyển ngàn...",| ma_nganh | ten_nganh | nam | diem | khoa |\n...,[ ma_nganh ...,[],text2sql,14.198,None
1,1,ablation_no_qe,Điểm chuẩn ngành Công nghệ Kỹ thuật ô tô (đào ...,"Theo nguồn tin, điểm chuẩn của ngành Công nghệ...",| ma_nganh | ten_nganh | nam | diem | khoa |\n...,[ ma_nganh ...,[],text2sql,13.957,None
2,2,ablation_no_qe,"Năm 2024, ngành Công nghệ thông tin (CTĐT Tiến...","Trong năm 2024, điểm trúng tuyển của ngành Côn...",| ma_nganh | ten_nganh ...,[ ma_nganh ten...,[],text2sql,16.205,None
3,3,ablation_no_qe,Xin cho biết điểm chuẩn ngành Robot và trí tuệ...,Điểm trúng tuyển vào ngành Robot và trí tuệ nh...,| ma_nganh | ten_nganh | diem | khoa |\n|---|-...,[ ma_nganh ...,[],text2sql,16.112,None
4,4,ablation_no_qe,Ngành Thương mại điện tử (đào tạo bằng tiếng V...,"Theo dữ liệu, mức điểm trúng tuyển của ngành T...",| Ngành | Năm | Điểm chuẩn |\n|---|---:|---:|\...,[ ma_nganh ...,[],text2sql,14.566,None


In [12]:
results_df_diemchuan = parse_list_columns(pd.read_csv(OUTPUT_PATH_DIEMCHUAN_ABLATION))
print(f"Loaded {len(results_df_diemchuan)} rows from Round 1")
print(f"Semantic cache has {semantic_cache.size} entries")

cached_latencies = []
cache_hits = 0
cache_misses = 0

for _, row in tqdm(results_df_diemchuan.iterrows(), total=len(results_df_diemchuan), desc="Round 2 – Cache"):
    question = row["question"]

    t0 = time.perf_counter()
    cached = await semantic_cache.search(question)
    latency_cached = time.perf_counter() - t0

    if cached is not None:
        cache_hits += 1
    else:
        cache_misses += 1

    cached_latencies.append(round(latency_cached, 3))

results_df_diemchuan["latency_cached_s"] = cached_latencies

serialize_list_columns(results_df_diemchuan).to_csv(OUTPUT_PATH_DIEMCHUAN_ABLATION, index=False, encoding="utf-8-sig")
print(f"\nRound 2 done – updated {OUTPUT_PATH_DIEMCHUAN_ABLATION}")
print(f"Cache hits: {cache_hits}, misses: {cache_misses}")
results_df_diemchuan[["pipeline", "tool_used", "latency_s", "latency_cached_s"]].head(10)

Loaded 200 rows from Round 1
Semantic cache has 200 entries


Round 2 – Cache:   0%|          | 0/200 [00:00<?, ?it/s]


Round 2 done – updated output/ablation_query_expansion_diemchuan.csv
Cache hits: 200, misses: 0


,pipeline,tool_used,latency_s,latency_cached_s
0,ablation_no_qe,text2sql,14.198,0.409
1,ablation_no_qe,text2sql,13.957,0.204
2,ablation_no_qe,text2sql,16.205,0.200
3,ablation_no_qe,text2sql,16.112,0.199
4,ablation_no_qe,text2sql,14.566,0.192
5,ablation_no_qe,text2sql,13.789,0.195
6,ablation_no_qe,text2sql,17.478,0.314
7,ablation_no_qe,text2sql,15.886,0.201
8,ablation_no_qe,text2sql,17.677,0.197
9,ablation_no_qe,text2sql,18.032,0.193


## Tạo evaluation dataset cho tuyển sinh

In [13]:
print(f"Loaded {len(df_tuyensinh)} questions from {FULL_DATASET_PATH_TUYENSINH}")

corpus_df = pd.read_csv(CORPUS_PATH)
corpus_lookup = {idx: row["name"] + "\n" + row["full_text"] for idx, row in corpus_df.iterrows()}
print(f"Loaded {len(corpus_lookup)} corpus documents for doc_id → full text lookup")

semantic_cache.clear()

results = []

for pipe_key, pipe in pipelines.items():
    label = PIPELINE_LABELS[pipe_key]
    print(f"\n{'='*60}")
    print(f"[Round 1 – Cold] Running pipeline: {label}")
    print(f"{'='*60}")

    for idx, row in tqdm(df_tuyensinh.iterrows(), total=len(df_tuyensinh), desc=label):
        question = row["question"]
        ground_truth = row["answer"]

        t0 = time.perf_counter()
        try:
            result = await pipe.run(question)
            latency = time.perf_counter() - t0
            answer = result.answer
            context = result.context
            doc_ids = result.doc_ids
            tool_used = result.tool_used
        except Exception as e:
            latency = time.perf_counter() - t0
            answer = f"ERROR: {e}"
            context = []
            doc_ids = []
            tool_used = "error"

        unique_ids = list(dict.fromkeys(doc_ids))
        retrieved_doc_contents = [
            corpus_lookup[int(did)] for did in unique_ids if int(did) in corpus_lookup
        ]

        await semantic_cache.add(question, answer)

        results.append({
            "qid": idx,
            "pipeline": pipe_key,
            "question": question,
            "ground_truth_answer": ground_truth,
            "generated_answer": answer,
            "context": context,
            "retrieved_doc_contents": retrieved_doc_contents,
            "tool_used": tool_used,
            "latency_s": round(latency, 3),
            "latency_cached_s": None,
        })

results_df_tuyensinh = pd.DataFrame(results)
os.makedirs("output", exist_ok=True)

serialize_list_columns(results_df_tuyensinh).to_csv(OUTPUT_PATH_TUYENSINH_ABLATION, index=False, encoding="utf-8")
print(f"\nRound 1 done – saved {len(results_df_tuyensinh)} rows to {OUTPUT_PATH_TUYENSINH_ABLATION}")
print(f"Semantic cache populated with {semantic_cache.size} entries")
results_df_tuyensinh.head()

Loaded 400 questions from dataset/evaluation_tuyensinh.csv
Loaded 498 corpus documents for doc_id → full text lookup

[Round 1 – Cold] Running pipeline: − Query Expansion


− Query Expansion:   0%|          | 0/400 [00:00<?, ?it/s]


Round 1 done – saved 400 rows to output/ablation_query_expansion_tuyensinh.csv
Semantic cache populated with 400 entries


,qid,pipeline,question,ground_truth_answer,generated_answer,context,retrieved_doc_contents,tool_used,latency_s,latency_cached_s
0,0,ablation_no_qe,Trường mình tên đầy đủ là gì vậy ạ?,Tên đầy đủ của trường mình là Trường Đại học S...,Trường Đại học Sư phạm Kỹ thuật Thành phố Hồ C...,[THÔNG TIN CHUNG\n10 lý do để bạn nên theo học...,[THÔNG TIN CHUNG\n10 lý do để bạn nên theo học...,document_search,7.847,None
1,1,ablation_no_qe,Địa chỉ chính xác của trường ở đâu thế ạ?,"Trường tọa lạc tại số 01, Võ Văn Ngân, Phường ...","Số 01 Võ Văn Ngân, Phường Linh Chiểu, TP. Thủ ...",[THÔNG TIN CHUNG\nGiới thiệu chung các kênh th...,[THÔNG TIN CHUNG\nGiới thiệu chung các kênh th...,document_search,11.483,None
2,2,ablation_no_qe,Trường được thành lập vào ngày nào vậy ạ?,Trường chúng ta chính thức được thành lập vào ...,05/10/1962,[THÔNG TIN CHUNG\nLịch sử hình thành và phát t...,[THÔNG TIN CHUNG\nLịch sử hình thành và phát t...,document_search,11.116,None
3,3,ablation_no_qe,Tiền thân của trường mình là đơn vị nào?,Tiền thân của trường mình chính là Ban Cao đẳn...,Ban Cao đẳng Sư phạm Kỹ thuật,[THÔNG TIN CHUNG\nLịch sử hình thành và phát t...,[THÔNG TIN CHUNG\nLịch sử hình thành và phát t...,document_search,6.395,None
4,4,ablation_no_qe,Giá trị cốt lõi của trường gồm những gì ạ?,Các giá trị cốt lõi của trường bao gồm Nhân bả...,- Gìn giữ và phát huy các giá trị truyền thống...,[THÔNG TIN CHUNG\nGiá trị cốt lõi\nTrường Đại ...,[THÔNG TIN CHUNG\nGiá trị cốt lõi\nTrường Đại ...,document_search,13.873,None


In [14]:
results_df_tuyensinh = parse_list_columns(pd.read_csv(OUTPUT_PATH_TUYENSINH_ABLATION))
print(f"Loaded {len(results_df_tuyensinh)} rows from Round 1")
print(f"Semantic cache has {semantic_cache.size} entries")

cached_latencies = []
cache_hits = 0
cache_misses = 0

for _, row in tqdm(results_df_tuyensinh.iterrows(), total=len(results_df_tuyensinh), desc="Round 2 – Cache"):
    question = row["question"]

    t0 = time.perf_counter()
    cached = await semantic_cache.search(question)
    latency_cached = time.perf_counter() - t0

    if cached is not None:
        cache_hits += 1
    else:
        cache_misses += 1

    cached_latencies.append(round(latency_cached, 3))

results_df_tuyensinh["latency_cached_s"] = cached_latencies

serialize_list_columns(results_df_tuyensinh).to_csv(OUTPUT_PATH_TUYENSINH_ABLATION, index=False, encoding="utf-8-sig")
print(f"\nRound 2 done – updated {OUTPUT_PATH_TUYENSINH_ABLATION}")
print(f"Cache hits: {cache_hits}, misses: {cache_misses}")
results_df_tuyensinh[["pipeline", "tool_used", "latency_s", "latency_cached_s"]].head(10)

Loaded 400 rows from Round 1
Semantic cache has 400 entries


Round 2 – Cache:   0%|          | 0/400 [00:00<?, ?it/s]


Round 2 done – updated output/ablation_query_expansion_tuyensinh.csv
Cache hits: 400, misses: 0


,pipeline,tool_used,latency_s,latency_cached_s
0,ablation_no_qe,document_search,7.847,0.279
1,ablation_no_qe,document_search,11.483,0.201
2,ablation_no_qe,document_search,11.116,0.207
3,ablation_no_qe,document_search,6.395,0.198
4,ablation_no_qe,document_search,13.873,0.200
5,ablation_no_qe,document_search,5.986,0.197
6,ablation_no_qe,document_search,10.885,0.197
7,ablation_no_qe,document_search,8.898,0.204
8,ablation_no_qe,document_search,8.328,0.200
9,ablation_no_qe,document_search,7.758,0.196


In [10]:
# test_df = pd.read_csv("output/evaluation_results_tuyensinh.csv")
# test_df.to_excel("test_tuyensinh_eva.xlsx", index=False)

In [11]:
# test_df = pd.read_excel("test_tuyensinh_eva.xlsx")
# test_df.to_csv("output/evaluation_results_tuyensinh.csv", index=False, encoding='utf-8-sig')